# MiniMind 大图景(精简版)

> 本文是 [ch01.ipynb](./ch01.ipynb) 的浓缩版。只保留核心代码,方便快速复习。

## 一个 LLM 的 6 个阶段

```
文本 →① Tokenizer→ ② Model(Transformer)→ ③ Pretrain → ④ SFT → ⑤ Alignment(DPO/PPO/GRPO)→ ⑥ Inference
```

minimind 覆盖全部 6 个阶段,本教程分 15 章逐层拆解。

## 最小推理代码

```python
import torch
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
model = MiniMindForCausalLM(config)
model.load_state_dict(torch.load('./out/full_sft_768.pth', map_location='cpu'), strict=True)
model = model.half().eval()  # 推理用 half;训练保持 float32 + bfloat16 autocast

tokenizer = AutoTokenizer.from_pretrained('./model')
conversation = [{'role': 'user', 'content': '你好'}]
text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt')

out = model.generate(inputs['input_ids'], max_new_tokens=64, do_sample=True,
                     temperature=0.85, top_p=0.95, pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
```

## 关键数字(minimind-3)

| 项 | 值 |
|---|---|
| 参数量 | ~64M(1/2700 of GPT-3) |
| 训练成本 | ~¥3(单 3090,2 小时 1 epoch SFT) |
| 词表大小 | 6400(BPE + ByteLevel) |
| 模型代码 | 288 行(单文件) |
| 架构对齐 | Qwen3 / Qwen3-MoE |

## 推理 4 步拆解

1. **tokenizer**:文本 → token id(`AutoTokenizer.encode`)
2. **model**:token id → logits(embed → 8×Block → lm_head)
3. **sampling**:logits → 下一个 token id(temperature + top_p)
4. **chat_template**:多轮对话格式化(`<|im_start|>role\n...<|im_end|>`)